# FlashEats - Workflow model and business metrics

This notebook builds the canonical one-row-per-order `order_journey` model. One-to-many sources are aggregated before joining so they cannot multiply the order grain.

In [ ]:
from pathlib import Path
import json
import sqlite3
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'data' / 'raw' / 'sqlite' / 'flasheats.db').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
FILES = RAW / 'files'
with sqlite3.connect(RAW / 'sqlite' / 'flasheats.db') as con:
    orders = pd.read_sql('select * from orders', con)
    customers = pd.read_sql('select * from customers', con)
    restaurants = pd.read_sql('select * from restaurants', con)
    drivers = pd.read_sql('select * from drivers', con)

def read_csv(name):
    return pd.read_csv(FILES / name)

outcomes = read_csv('order_outcomes.csv')
app_actions = read_csv('customer_app_actions.csv')
interactions = read_csv('customer_interactions.csv')
tickets = read_csv('support_tickets.csv')
interventions = read_csv('order_interventions.csv')
order_events = read_csv('order_events.csv')
restaurant_status = read_csv('restaurant_status.csv')
dispatch = pd.DataFrame(json.loads((RAW / 'api' / 'source' / 'dispatch_data.json').read_text(encoding='utf-8')))
driver_objects = json.loads((FILES / 'driver_events.json').read_text(encoding='utf-8'))
driver_events = pd.DataFrame([{'driver_id': item['driver_id'], **event} for item in driver_objects for event in item.get('events', [])])

for column in ['created_at', 'promised_eta', 'pickup_at', 'actual_delivery_at']:
    orders[column] = pd.to_datetime(orders[column], errors='coerce')
orders = orders.drop_duplicates('order_id', keep='first').copy()
print('canonical order grain:', len(orders), 'orders')

## 1. Aggregate events, interactions, interventions, and customer friction

In [ ]:
def count_by_order(frame, key, output_name):
    return frame.groupby(key).size().rename(output_name).reset_index()

app_summary = (app_actions.groupby('order_id').agg(app_action_count=('action_id', 'count'), cancel_attempted=('action_type', lambda s: int(s.eq('CANCEL_ATTEMPTED').any())), support_opened=('action_type', lambda s: int(s.eq('SUPPORT_OPENED').any()))).reset_index())
interaction_summary = (interactions.groupby('order_id').agg(interaction_count=('interaction_id', 'count'), support_interaction=('interaction_type', lambda s: int(s.eq('SUPPORT_TICKET').any()))).reset_index())
ticket_summary = (tickets[tickets.order_id.notna() & tickets.order_id.astype(str).str.strip().ne('')].groupby('order_id').agg(ticket_count=('ticket_id', 'count'), ticket_opened=('ticket_id', 'size')).reset_index())
intervention_summary = (interventions.groupby('order_id').agg(intervention_count=('intervention_id', 'count'), intervention_types=('intervention_type', lambda s: ','.join(sorted(set(s))))).reset_index())
event_summary = (order_events.groupby('order_id').agg(event_count=('event_id', 'count'), reassignment_event_count=('event_type', lambda s: int(s.eq('DRIVER_REASSIGNMENT').sum()))).reset_index())
status_summary = (restaurant_status.assign(status_norm=restaurant_status.status.astype(str).str.strip().str.lower()).groupby('order_id').agg(restaurant_status_updates=('status', 'count'), last_restaurant_status=('status_norm', 'last')).reset_index())

journey = orders.merge(outcomes, on='order_id', how='left', suffixes=('', '_outcome'))
for summary in [app_summary, interaction_summary, ticket_summary, intervention_summary, event_summary, status_summary]:
    journey = journey.merge(summary, on='order_id', how='left')
for column in ['app_action_count', 'interaction_count', 'ticket_count', 'intervention_count', 'event_count', 'restaurant_status_updates', 'reassignment_event_count']:
    if column in journey:
        journey[column] = journey[column].fillna(0).astype(int)
for column in ['cancel_attempted', 'support_opened', 'support_interaction', 'ticket_opened']:
    if column in journey:
        journey[column] = journey[column].fillna(0).astype(int)

journey['order_to_pickup_min'] = (journey.pickup_at - journey.created_at).dt.total_seconds() / 60
journey['transit_min'] = (journey.actual_delivery_at - journey.pickup_at).dt.total_seconds() / 60
journey = journey.merge(restaurants[['restaurant_id', 'cuisine']], on='restaurant_id', how='left')
journey = journey.merge(drivers[['driver_id', 'vehicle_type', 'experience_months']], on='driver_id', how='left')
journey['has_customer_friction'] = ((journey.get('support_opened', 0) > 0) | (journey.get('ticket_opened', 0) > 0) | (journey.get('support_interaction', 0) > 0)).astype(int)
journey['has_intervention'] = (journey['intervention_count'] > 0).astype(int)
print('journey rows:', len(journey), 'unique orders:', journey.order_id.nunique())
display(journey.head())

## 2. KPI-linked metrics

The outcome source supplies the KPI denominator. Associations are descriptive and do not establish causal impact.

In [ ]:
measurable = journey[journey['delay_min'].notna()].copy()
measurable['delay_min'] = pd.to_numeric(measurable['delay_min'], errors='coerce')
measurable['late_flag'] = pd.to_numeric(measurable['late_flag'], errors='coerce')

metrics = pd.DataFrame([
    ['late_delivery_rate_pct', measurable.late_flag.eq(1).mean() * 100, '843 / 1,495 measurable delivered outcomes', 'Outcome KPI'],
    ['median_delay_among_late_min', measurable.loc[measurable.late_flag.eq(1), 'delay_min'].median(), 'Late deliveries only', 'Severity'],
    ['median_order_to_pickup_min', journey.order_to_pickup_min.median(), 'Orders with pickup timestamp', 'Restaurant/preparation proxy'],
    ['customer_friction_rate_pct', measurable.has_customer_friction.mean() * 100, 'Orders with support/app friction signal', 'Customer experience'],
    ['intervention_rate_pct', measurable.has_intervention.mean() * 100, 'Orders with at least one intervention', 'Operational response'],
], columns=['metric', 'value', 'denominator_note', 'role'])
metrics['value'] = metrics['value'].round(2)
display(metrics)

comparison = measurable.groupby('has_intervention').agg(orders=('order_id', 'size'), late_rate_pct=('late_flag', lambda s: round(s.mean() * 100, 2))).reset_index()
comparison['group'] = comparison.has_intervention.map({0: 'No intervention', 1: 'Intervention'})
display(comparison[['group', 'orders', 'late_rate_pct']])

## 3. Save the analytical model and evidence output

These outputs are reproducible artifacts; the documentation phase will explain their ownership, limitations, and publication rules.

In [ ]:
OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)
journey.to_csv(OUTPUTS / 'order_journey.csv', index=False)
metrics.to_csv(OUTPUTS / 'metric_evidence.csv', index=False)
comparison.to_csv(OUTPUTS / 'intervention_comparison.csv', index=False)
print('Saved:', OUTPUTS / 'order_journey.csv', OUTPUTS / 'metric_evidence.csv', OUTPUTS / 'intervention_comparison.csv')